# 02. Exploratory Data Analysis & Visual Intelligence
## Project: AI-Driven Loan Recovery & Risk Analytics

### Overview
This notebook presents in-depth exploratory data analysis across credit risk and debt recovery dimensions:
1. **Portfolio Distributions**: Loan amounts, interest rates, credit scores, and debt-to-income.
2. **Delinquency Analysis**: Overdue days by product type, geography, and sourcing channel.
3. **Recovery Channel Efficiency**: ROI multipliers and recovery rates across AI bots, tele-calling, field visits, and legal.
4. **Correlation Analysis**: Multi-variable financial risk correlation matrix.
5. **Vintage Cohort Curves**: Cumulative repayment curves across loan origination quarters.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

df = pd.read_csv("../data/processed/loan_recovery_master.csv")
delinquent_df = df[df["overdue_days"] > 30].copy()
print(f"Loaded Master Analytical Dataset: {df.shape}")


### 1. Macro Portfolio Distribution & Risk Profile

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x="credit_score", hue="delinquency_bucket", multiple="stack", ax=axes[0], palette="turbo")
axes[0].set_title("Credit Score Distribution by Delinquency Bucket", fontweight="bold")

sns.boxplot(data=df, x="loan_type", y="interest_rate", ax=axes[1], palette="Set2")
axes[1].set_title("Interest Rate by Loan Product Line", fontweight="bold")
plt.tight_layout()
plt.show()


### 2. Recovery Rate & Efficiency by Collection Channel

In [ ]:
rec_chan = delinquent_df.groupby("primary_channel").agg(
    outstanding=("outstanding_principal", "sum"),
    recovered=("recovered_amount", "sum"),
    cost=("recovery_cost", "sum")
).reset_index()
rec_chan["recovery_rate_%"] = (rec_chan["recovered"] / rec_chan["outstanding"]) * 100
rec_chan["roi_multiple"] = rec_chan["recovered"] / np.maximum(rec_chan["cost"], 1.0)
display(rec_chan.sort_values("recovery_rate_%", ascending=False))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=rec_chan.sort_values("recovery_rate_%", ascending=False), x="recovery_rate_%", y="primary_channel", palette="Greens_r", ax=ax)
ax.set_title("Recovery Rate (%) by Channel", fontweight="bold")
for p in ax.patches:
    ax.annotate(f"{p.get_width():.1f}%", (p.get_width() + 0.5, p.get_y() + p.get_height() / 2.), va="center")
plt.show()


### 3. Correlation Matrix of Financial & Risk Drivers

In [ ]:
plt.figure(figsize=(11, 8))
cols = ["annual_income", "credit_score", "loan_amount", "interest_rate", "dti_ratio", "overdue_days", "recovered_amount", "default_flag"]
sns.heatmap(df[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Matrix of Key Credit & Recovery Drivers", fontweight="bold")
plt.show()
